CALIBRATION 1

In [1]:
import os
import time
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from datasets import Dataset
from datetime import timedelta
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback, 
    TrainerCallback
)

In [2]:
CALIBRATED_DATA = r'D:\Major Project\SpamX\machine_learning\dataset\calibrated.csv'
VAL_DATA = r'D:\Major Project\SpamX\machine_learning\dataset\validation.csv'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
models_to_refine = [
    {"name": "muril", "path": r"D:\Major Project\SpamX\machine_learning\models\MuRIL\final_muril"},
    {"name": "xlm_roberta", "path": r"D:\Major Project\SpamX\machine_learning\models\XLM_Roberta\final_xlm_roberta"},
    {"name": "indicbert", "path": r"D:\Major Project\SpamX\machine_learning\models\IndicBERT\final_indicbert"},
    {"name": "mbert", "path": r"D:\Major Project\SpamX\machine_learning\models\mBERT\final_mbert"}
]

In [4]:
class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        gamma, alpha = 2.0, 0.25 
        probs = F.softmax(logits, dim=-1)
        pt = probs.gather(1, labels.unsqueeze(1)).squeeze(1)
        loss = -alpha * (1 - pt) ** gamma * (pt + 1e-8).log()
        return (loss.mean(), outputs) if return_outputs else loss.mean()

In [5]:
class ETALoggingCallback(TrainerCallback):
    def __init__(self, model_name):
        self.start_time = None
        self.model_name = model_name

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f"\n{self.model_name.upper()} Finetuning Started")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.start_time and state.global_step > 0:
            elapsed = time.time() - self.start_time
            avg_time_per_step = elapsed / state.global_step
            remaining_steps = state.max_steps - state.global_step
            eta_str = str(timedelta(seconds=int(remaining_steps * avg_time_per_step)))
            print(f"Step: {state.global_step}/{state.max_steps} | ETA: {eta_str} | Progress: {(state.global_step/state.max_steps)*100:.2f}%")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions)
    }

In [6]:
def get_tokenize_func(tokenizer):
    def tokenize(examples):
        tokenized = tokenizer(
            [str(x) for x in examples["Comment"]],
            truncation=True, max_length=128, stride=32,
            return_overflowing_tokens=True, padding="max_length"
        )
        sample_mapping = tokenized.pop("overflow_to_sample_mapping")
        tokenized["labels"] = [examples["Label"][i] for i in sample_mapping]
        return tokenized
    return tokenize

In [7]:
train_df = pd.read_csv(CALIBRATED_DATA)
val_df = pd.read_csv(VAL_DATA)

In [8]:
for m_info in models_to_refine:
    tokenizer = AutoTokenizer.from_pretrained(m_info['path'], trust_remote_code=True)
    model = AutoModelForSequenceClassification.from_pretrained(m_info['path'], num_labels=2, trust_remote_code=True).to(DEVICE)
    
    tok_func = get_tokenize_func(tokenizer)
    train_ds = Dataset.from_pandas(train_df).map(tok_func, batched=True, remove_columns=train_df.columns.tolist())
    val_ds = Dataset.from_pandas(val_df).map(tok_func, batched=True, remove_columns=val_df.columns.tolist())
    
    refine_args = TrainingArguments(
        output_dir=f"./temp_{m_info['name']}",
        num_train_epochs=2,
        per_device_train_batch_size=16,
        learning_rate=5e-6, 
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=True,
        report_to="none"
    )
    
    trainer = FocalLossTrainer(
        model=model,
        args=refine_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        callbacks=[ETALoggingCallback(m_info['name'])]
    )
    
    trainer.train()

    save_path = f"{m_info['path']}_calibrated1"
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Saved: {save_path}")

Map:   0%|          | 0/739 [00:00<?, ? examples/s]

Map:   0%|          | 0/6438 [00:00<?, ? examples/s]


MURIL Finetuning Started


  0%|          | 0/94 [00:00<?, ?it/s]

  0%|          | 0/814 [00:00<?, ?it/s]

...

  0%|          | 0/96 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

Step: 48/96 | ETA: 0:00:32 | Progress: 50.00%
{'eval_loss': 0.006449957378208637, 'eval_accuracy': 0.9692142747740848, 'eval_f1': 0.9691954022988506, 'eval_precision': 0.9613864396473092, 'eval_recall': 0.977132262051916, 'eval_runtime': 24.7101, 'eval_samples_per_second': 264.224, 'eval_steps_per_second': 33.063, 'epoch': 1.0}


  0%|          | 0/817 [00:00<?, ?it/s]

Step: 96/96 | ETA: 0:00:00 | Progress: 100.00%
{'eval_loss': 0.006923942361027002, 'eval_accuracy': 0.9672231582171849, 'eval_f1': 0.9672882910424947, 'eval_precision': 0.957047791893527, 'eval_recall': 0.9777503090234858, 'eval_runtime': 26.0236, 'eval_samples_per_second': 250.888, 'eval_steps_per_second': 31.395, 'epoch': 2.0}
Step: 96/96 | ETA: 0:00:00 | Progress: 100.00%
{'train_runtime': 72.4045, 'train_samples_per_second': 20.827, 'train_steps_per_second': 1.326, 'train_loss': 0.012629929929971695, 'epoch': 2.0}
Saved: D:\Major Project\SpamX\machine_learning\models\mBERT\final_mbert_calibrated1


Calibration 2 (MuRIL and XLM RoBERTa)

In [10]:
import os
import time
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from datasets import Dataset
from datetime import timedelta
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    TrainerCallback
)

In [11]:
CALIBRATED_DATA = r'D:\Major Project\SpamX\machine_learning\dataset\calibrated1.csv'
VAL_DATA = r'D:\Major Project\SpamX\machine_learning\dataset\validation.csv'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [12]:
models_to_refine = [
    {"name": "muril", "path": r"D:\Major Project\SpamX\machine_learning\models\MuRIL\final_muril_calibrated1"},
    {"name": "xlm_roberta", "path": r"D:\Major Project\SpamX\machine_learning\models\XLM_Roberta\final_xlm_roberta_calibrated1"}
]

In [13]:
class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        gamma, alpha = 2.0, 0.25 
        probs = F.softmax(logits, dim=-1)
        pt = probs.gather(1, labels.unsqueeze(1)).squeeze(1)
        loss = -alpha * (1 - pt) ** gamma * (pt + 1e-8).log()
        return (loss.mean(), outputs) if return_outputs else loss.mean()

In [14]:
class ETALoggingCallback(TrainerCallback):
    def __init__(self, model_name):
        self.start_time = None
        self.model_name = model_name

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f"\n{self.model_name.upper()} Incremental Fine-tuning Started")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.start_time and state.global_step > 0:
            elapsed = time.time() - self.start_time
            avg_time_per_step = elapsed / state.global_step
            remaining_steps = state.max_steps - state.global_step
            eta_str = str(timedelta(seconds=int(remaining_steps * avg_time_per_step)))
            print(f"🔹 Step: {state.global_step}/{state.max_steps} | ETA: {eta_str}")

In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions)
    }

In [16]:
def get_tokenize_func(tokenizer):
    def tokenize(examples):
        tokenized = tokenizer(
            [str(x) for x in examples["Comment"]],
            truncation=True, 
            max_length=128, 
            stride=32,
            return_overflowing_tokens=True, 
            padding="max_length"
        )
        sample_mapping = tokenized.pop("overflow_to_sample_mapping")
        tokenized["labels"] = [examples["Label"][i] for i in sample_mapping]
        return tokenized
    return tokenize

In [17]:
train_df = pd.read_csv(CALIBRATED_DATA)
val_df = pd.read_csv(VAL_DATA)
train_df['Label'] = train_df['Label'].astype(int)
val_df['Label'] = val_df['Label'].astype(int)

In [ ]:
for m_info in models_to_refine:
    print(f"\n--- Refining {m_info['name']} ---")
    
    tokenizer = AutoTokenizer.from_pretrained(m_info['path'], trust_remote_code=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        m_info['path'], num_labels=2, trust_remote_code=True
    ).to(DEVICE)
    
    tok_func = get_tokenize_func(tokenizer)
    
    train_ds = Dataset.from_pandas(train_df).map(tok_func, batched=True, remove_columns=train_df.columns.tolist())
    val_ds = Dataset.from_pandas(val_df).map(tok_func, batched=True, remove_columns=val_df.columns.tolist())
    
    refine_args = TrainingArguments(
    output_dir=f"./temp_inc_{m_info['name']}",
    num_train_epochs=5,              
    per_device_train_batch_size=12,
    gradient_accumulation_steps=4,
    evaluation_strategy="steps",     
    eval_steps=100,                  
    save_strategy="steps",           
    save_steps=100,
    load_best_model_at_end=True,     
    metric_for_best_model="f1",      
    fp16=True,
    report_to="none"
)
    
    trainer = FocalLossTrainer(
        model=model,
        args=refine_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        callbacks=[
            ETALoggingCallback(m_info['name']),
            EarlyStoppingCallback(early_stopping_patience=3)
        ]
    )
    
    trainer.train()

    save_path = f"{m_info['path']}_v2"
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    
    del model, tokenizer, trainer
    torch.cuda.empty_cache()
    print(f"Saved Final Model: {save_path}")


--- Refining muril ---


Map:   0%|          | 0/52156 [00:00<?, ? examples/s]

Map:   0%|          | 0/6438 [00:00<?, ? examples/s]


MURIL Incremental Fine-tuning Started


  0%|          | 0/5495 [00:00<?, ?it/s]

...

  0%|          | 0/815 [00:00<?, ?it/s]

🔹 Step: 300/5505 | ETA: 3:40:48
{'eval_loss': 0.0057188766077160835, 'eval_accuracy': 0.9687068568798896, 'eval_f1': 0.9675572519083969, 'eval_precision': 0.9963969865705863, 'eval_recall': 0.9403400309119011, 'eval_runtime': 26.0329, 'eval_samples_per_second': 250.414, 'eval_steps_per_second': 31.307, 'epoch': 0.27}


  0%|          | 0/815 [00:00<?, ?it/s]

🔹 Step: 400/5505 | ETA: 3:35:56
{'eval_loss': 0.005107122007757425, 'eval_accuracy': 0.972541800889707, 'eval_f1': 0.9721487474716042, 'eval_precision': 0.9786967418546366, 'eval_recall': 0.9656877897990727, 'eval_runtime': 24.4851, 'eval_samples_per_second': 266.244, 'eval_steps_per_second': 33.286, 'epoch': 0.36}
🔹 Step: 400/5505 | ETA: 3:37:33
{'train_runtime': 1022.788, 'train_samples_per_second': 258.304, 'train_steps_per_second': 5.382, 'train_loss': 0.007559279203414917, 'epoch': 0.36}
Saved Final Model: D:\Major Project\SpamX\machine_learning\models\XLM_Roberta\final_xlm_roberta_calibrated1_v2


: 